In [1]:
import os
os.chdir("..")

from utils.utils import notebook_line_magic
notebook_line_magic()

Line Magic Set


In [26]:
from config.exp_config import get_mujoco_exp_names_and_replay_paths

exp_names, replay_paths = get_mujoco_exp_names_and_replay_paths(baseline="cql")

In [25]:
# str(exp_names[0])
exp_names

[PosixPath('generated_artifacts/halfcheetah_random_cql_seed5'),
 PosixPath('generated_artifacts/halfcheetah_random_cql_seed10'),
 PosixPath('generated_artifacts/halfcheetah_random_cql_seed20'),
 PosixPath('generated_artifacts/halfcheetah_random_cql_seed30')]

## CQL

In [6]:
import warnings
warnings.simplefilter('ignore')

from utils.utils import set_ld_library_path
from ding.entry import serial_pipeline_offline
from dizoo.d4rl.config.halfcheetah_random_cql_config import main_config, create_config
from ding.config import compile_config

set_ld_library_path()

MUJOCO_GL = osmesa
MUJOCO_PY_MUJOCO_PATH = /home/azm0269@auburn.edu/.mujoco/mujoco210
LD_LIBRARY_PATH entries:
   /home/azm0269@auburn.edu/.mujoco/mujoco210/bin
   /home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mujoco_py/generated/_pyxbld_2.1.2.14_310_linuxcpuextensionbuilder/lib.linux-x86_64-cpython-310/mujoco_py


In [3]:
main_config.exp_name = "generated_artifacts/halfcheetah_expert_cql_seed0"

In [26]:
main_config.env.env_id = "bullet-halfcheetah-expert-v2"
main_config.env.replay_path="./video/mujoco/halfcheetah"

In [27]:
cfg = compile_config(
    main_config,
    create_cfg=create_config,
    auto=True,
)

In [ ]:
# cfg.policy.model

# main_config
import gym
gym.envs.registry.keys()

dict_keys(['ALE/Tetris-v5', 'ALE/Tetris-ram-v5', 'Adventure-v0', 'AdventureDeterministic-v0', 'AdventureNoFrameskip-v0', 'Adventure-v4', 'AdventureDeterministic-v4', 'AdventureNoFrameskip-v4', 'Adventure-ram-v0', 'Adventure-ramDeterministic-v0', 'Adventure-ramNoFrameskip-v0', 'Adventure-ram-v4', 'Adventure-ramDeterministic-v4', 'Adventure-ramNoFrameskip-v4', 'AirRaid-v0', 'AirRaidDeterministic-v0', 'AirRaidNoFrameskip-v0', 'AirRaid-v4', 'AirRaidDeterministic-v4', 'AirRaidNoFrameskip-v4', 'AirRaid-ram-v0', 'AirRaid-ramDeterministic-v0', 'AirRaid-ramNoFrameskip-v0', 'AirRaid-ram-v4', 'AirRaid-ramDeterministic-v4', 'AirRaid-ramNoFrameskip-v4', 'Alien-v0', 'AlienDeterministic-v0', 'AlienNoFrameskip-v0', 'Alien-v4', 'AlienDeterministic-v4', 'AlienNoFrameskip-v4', 'Alien-ram-v0', 'Alien-ramDeterministic-v0', 'Alien-ramNoFrameskip-v0', 'Alien-ram-v4', 'Alien-ramDeterministic-v4', 'Alien-ramNoFrameskip-v4', 'Amidar-v0', 'AmidarDeterministic-v0', 'AmidarNoFrameskip-v0', 'Amidar-v4', 'AmidarDete

In [28]:
import gym
from dizoo.d4rl.envs.d4rl_env import D4RLEnv
from ding.envs import DingEnvWrapper, BaseEnvManagerV2

# collector_env = BaseEnvManagerV2(
#     env_fn=[lambda: DingEnvWrapper(gym.make("HalfCheetah-v2")) for _ in range(cfg.env.collector_env_num)],
#     cfg=cfg.env.manager
# )
# evaluator_env = BaseEnvManagerV2(
#     env_fn=[lambda: DingEnvWrapper(gym.make("HalfCheetah-v2")) for _ in range(cfg.env.evaluator_env_num)],
#     cfg=cfg.env.manager
# )
# cfg.env

In [29]:
# serial_pipeline_offline([main_config, create_config], seed=0)
from ding.model import ContinuousQAC
from ding.policy import CQLPolicy
from ding.data import DequeBuffer

model = ContinuousQAC(**cfg.policy.model)
buffer_ = DequeBuffer(size=cfg.policy.other.replay_buffer.replay_buffer_size)
policy = CQLPolicy(cfg=cfg.policy, model=model)

In [8]:
# import 

In [30]:
# from d4rl.locomotion import ant
from ding.utils import set_pkg_seed
from ding.framework import task, ding_init
from ding.data import create_dataset
from ding.framework.context import OfflineRLContext
from ding.framework.middleware import interaction_evaluator, trainer, CkptSaver, offline_data_fetcher, offline_logger

ding_init(cfg)
with task.start(async_mode=False, ctx=OfflineRLContext()):
    # Evaluating, we place it on the first place to get the score of the random model as a benchmark value
    evaluator_env = BaseEnvManagerV2(
        env_fn=[lambda: D4RLEnv(cfg.env) for _ in range(cfg.env.evaluator_env_num)], cfg=cfg.env.manager
    )
    set_pkg_seed(cfg.seed, use_cuda=cfg.policy.cuda)
    
    dataset = create_dataset(cfg)
    model = ContinuousQAC(**cfg.policy.model)
    policy = CQLPolicy(cfg.policy, model=model)
    
    task.use(interaction_evaluator(cfg, policy.eval_mode, evaluator_env))
    task.use(offline_data_fetcher(cfg, dataset))
    task.use(trainer(cfg, policy.learn_mode))
    task.use(CkptSaver(policy, cfg.exp_name, train_freq=100))
    task.use(offline_logger())
    task.run()

VersionNotFound: Environment version `v2` for environment `bullet-halfcheetah-expert` doesn't exist. It provides versioned environments: [ `v0` ].

In [8]:
cfg

{'env': {'manager': {'episode_num': inf,
   'max_retry': 1,
   'retry_type': 'reset',
   'auto_reset': True,
   'step_timeout': None,
   'reset_timeout': None,
   'retry_waiting_time': 0.1,
   'cfg_type': 'BaseEnvManagerDict',
   'type': 'base'},
  'stop_value': 6000,
  'n_evaluator_episode': 8,
  'type': 'd4rl',
  'import_names': ['dizoo.d4rl.envs.d4rl_env'],
  'env_id': 'halfcheetah-expert-v2',
  'collector_env_num': 1,
  'evaluator_env_num': 8,
  'use_act_scale': True},
 'policy': {'model': {'twin_critic': True,
   'action_space': 'reparameterization',
   'actor_head_hidden_size': 256,
   'critic_head_hidden_size': 256,
   'obs_shape': 17,
   'action_shape': 6},
  'learn': {'learner': {'train_iterations': 1000000000,
    'dataloader': {'num_workers': 0},
    'log_policy': True,
    'hook': {'load_ckpt_before_run': '',
     'log_show_after_iter': 100,
     'save_ckpt_after_iter': 10000,
     'save_ckpt_after_run': True},
    'cfg_type': 'BaseLearnerDict'},
   'resume_training': False